# 📓 Notebook 14 — Filesystem Agents---**Phase 5 · Advanced Deep Agent Concepts**> An agent that can read files, write code, and execute it — this is where things get powerful (and dangerous).### What You'll Learn| Topic | Why It Matters ||-------|---------------|| File read/write | Agent interacts with the filesystem || Code generation | LLM generates Python code || Sandbox execution | Run generated code safely || Safety controls | Prevent dangerous file operations |### 🏗️ Build: Mini AutoGPT-Style Coding Agent

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Safe Filesystem Operations> ⚠️ **CRITICAL:** Never let an LLM directly access your real filesystem without strict controls.

In [ ]:
class SafeFileSystem:    """Sandboxed filesystem operations — all paths restricted to sandbox directory."""        def __init__(self, sandbox_dir):        self.sandbox = os.path.abspath(sandbox_dir)        os.makedirs(self.sandbox, exist_ok=True)        def _safe_path(self, path):        """Ensure path is within sandbox."""        full = os.path.abspath(os.path.join(self.sandbox, path))        if not full.startswith(self.sandbox):            raise PermissionError(f"Access denied: path escapes sandbox: {path}")        return full        def read(self, path):        safe = self._safe_path(path)        if not os.path.exists(safe):            return json.dumps({"error": f"File not found: {path}"})        with open(safe, "r") as f:            return json.dumps({"content": f.read(), "path": path})        def write(self, path, content):        safe = self._safe_path(path)        os.makedirs(os.path.dirname(safe), exist_ok=True)        with open(safe, "w") as f:            f.write(content)        return json.dumps({"status": "written", "path": path, "bytes": len(content)})        def list_dir(self, path="."):        safe = self._safe_path(path)        if not os.path.exists(safe):            return json.dumps({"error": f"Directory not found: {path}"})        items = []        for item in os.listdir(safe):            full = os.path.join(safe, item)            items.append({                "name": item,                "type": "dir" if os.path.isdir(full) else "file",                "size": os.path.getsize(full) if os.path.isfile(full) else None,            })        return json.dumps({"items": items, "path": path})        def delete(self, path):        safe = self._safe_path(path)        if os.path.isfile(safe):            os.remove(safe)        elif os.path.isdir(safe):            shutil.rmtree(safe)        return json.dumps({"status": "deleted", "path": path})fs = SafeFileSystem(SANDBOX)# Testprint(fs.write("hello.txt", "Hello from the agent!"))print(fs.read("hello.txt"))print(fs.list_dir())

---## 3. Safe Code Execution

In [ ]:
class CodeSandbox:    """Execute Python code in a sandboxed subprocess."""        def __init__(self, timeout=10, max_output=5000):        self.timeout = timeout        self.max_output = max_output        def run(self, code, filename="script.py"):        """Execute Python code safely."""        # Write to temp file        script_path = os.path.join(SANDBOX, filename)        with open(script_path, "w") as f:            f.write(code)                try:            result = subprocess.run(                ["python", script_path],                capture_output=True, text=True,                timeout=self.timeout,                cwd=SANDBOX,            )                        output = result.stdout[:self.max_output]            error = result.stderr[:self.max_output] if result.returncode != 0 else ""                        return json.dumps({                "success": result.returncode == 0,                "output": output,                "error": error,                "return_code": result.returncode,            })        except subprocess.TimeoutExpired:            return json.dumps({"success": False, "error": f"Timeout after {self.timeout}s"})        except Exception as e:            return json.dumps({"success": False, "error": str(e)})sandbox = CodeSandbox(timeout=10)# Testresult = sandbox.run("print('Hello from sandboxed code!')\nprint(2 + 2)")print(json.loads(result))

In [ ]:
class CodingAgent:    """Agent that generates, runs, and debugs code."""        def __init__(self, model=None):        self.model = model or DEFAULT_MODEL        self.fs = SafeFileSystem(SANDBOX)        self.sandbox = CodeSandbox(timeout=10)        def code_task(self, task, max_attempts=3, verbose=True):        """Generate code for a task, run it, and fix errors."""                if verbose:            print(f"📋 Task: {task}")            print("=" * 50)                for attempt in range(1, max_attempts + 1):            # Generate code            gen_prompt = f"Write Python code to accomplish this task. Return ONLY the code, no explanations.\n\nTask: {task}"            if attempt > 1:                gen_prompt += f"\n\nPrevious attempt had this error:\n{last_error}\n\nFix the error and try again."                        response = litellm.completion(                model=self.model,                messages=[{                    "role": "system",                    "content": "You are a Python code generator. Return ONLY valid Python code. No markdown, no explanations."                }, {"role": "user", "content": gen_prompt}],                temperature=0.2, max_tokens=800,            )                        code = response.choices[0].message.content            # Strip markdown code fences if present            code = code.replace("```python", "").replace("```", "").strip()                        if verbose:                print(f"\n🔧 Attempt {attempt}:")                print(f"   Code ({len(code)} chars): {code[:100]}...")                        # Execute            result = json.loads(self.sandbox.run(code, f"task_attempt_{attempt}.py"))                        if result["success"]:                if verbose:                    print(f"   ✅ Success!")                    if result["output"]:                        print(f"   Output: {result['output'][:200]}")                return {"success": True, "code": code, "output": result["output"], "attempts": attempt}            else:                last_error = result["error"]                if verbose:                    print(f"   ❌ Error: {last_error[:100]}")                return {"success": False, "code": code, "error": last_error, "attempts": max_attempts}agent = CodingAgent()# Testtasks = [    "Calculate the first 20 Fibonacci numbers and print them",    "Generate a multiplication table for numbers 1-5 and print it nicely formatted",    "Write a function that checks if a string is a palindrome, test it with 'racecar', 'hello', 'madam'",]for task in tasks:    result = agent.code_task(task)    print("-" * 50)

---## 📝 Interview Quiz — Filesystem Agents### Q1: What are the security risks of a filesystem agent?<details><summary>💡 Answer</summary>1. **Path traversal**: Agent writes to `../../etc/passwd` or `C:\Windows\System32`2. **Arbitrary code execution**: Agent runs malicious code3. **Data exfiltration**: Agent reads sensitive files and sends them via API4. **Resource exhaustion**: Agent fills disk, runs infinite loops5. **Privilege escalation**: Agent runs as root/admin**Defenses:** Sandbox directory, path validation, timeout limits, output size limits, no network access in sandbox, run as unprivileged user.</details>### Q2: How would you sandbox code execution in production?<details><summary>💡 Answer</summary>Levels of isolation (weakest to strongest):1. **subprocess + timeout**: Basic (what we did here)2. **Docker container**: Process + filesystem isolation3. **gVisor / Firecracker**: Lightweight VM-level isolation4. **WebAssembly (WASM)**: Language-level sandboxing5. **Full VM**: Complete isolation (AWS Lambda approach)**Production recommendation:** Docker with no network, read-only mounts, resource limits (CPU, memory, disk), and timeout.</details>### Q3: How does an LLM debug its own generated code?<details><summary>💡 Answer</summary>Loop: Generate → Execute → Check result → If error, feed error back → Re-generateThe LLM receives:1. Original task description2. Previous code attempt3. Error message (stderr, traceback)The error message is often sufficient for the LLM to identify and fix the bug. This works because LLMs have seen millions of error messages and their fixes in training data.**Limiting factor:** Complex logical errors (wrong output, not errors) are harder to self-fix.</details>

---## ✅ Notebook 14 Summary| Concept | Key Takeaway ||---------|-------------|| Sandboxed filesystem | Restrict all paths to sandbox directory || Code generation | LLM writes Python, we execute in subprocess || Self-debugging | Feed errors back to LLM for automatic fixing || Safety layers | Path validation, timeout, output limits, no network || Production | Use Docker/WASM for real isolation |### ➡️ Next: [Notebook 15 — Evaluation](./15_evaluation.ipynb)